# EDGAR - The SEC's Electronic Data Gathering, Analysis, and Retrieval (EDGAR) system

There are a lot of EDGAR-related endpoints at https://www.sec.gov.  Most of the EDGAR overview can be found through [the link to search public filings](https://www.sec.gov/search-filings).  The EDGAR search itself is available at https://www.sec.gov/edgar/search/.  There is a good amount of additional info in the [FAQ](https://www.sec.gov/edgar/search/efts-faq.html), the [API documentation](https://www.sec.gov/search-filings/edgar-application-programming-interfaces), and the [data resources](https://www.sec.gov/data-research/sec-data-resources).

### Open government data in general

Slightly off-topic here, but there are a lot of additional open datasets at https://data.gov/open-gov/.

## Imports

In [2]:
import requests
import json
from io import StringIO
import datetime
import pandas as pd

## Headers

According to the [developer FAQ](https://www.sec.gov/about/webmaster-frequently-asked-questions#developers), you must declare your identity in the `User-Agent` header.

In [3]:
with open('../../../../../.config/edgar/user-agent', 'r') as file:
    user_agent = file.read().strip()
headers = {
    'User-Agent': user_agent
}

## CIK data

EDGAR's Central Index Key (CIK) is the uuid used to track filing registered by a particular entity.  There is a large list of these downloadable at https://www.sec.gov/Archives/edgar/cik-lookup-data.txt .

There are alernative (possibly old/out-dated) mappings by ticker https://www.sec.gov/files/company_tickers.json and by ticket and exchange https://www.sec.gov/files/company_tickers_exchange.json .

### All CIK data

In [4]:
# Todo - parse the txt data at ../../data/edgar/all_cik_data.txt

### CIK by ticker

In [5]:
cik_ticker = pd.read_json('../../data/edgar/company_tickers.json')
cik_ticker = cik_ticker.transpose()
cik_ticker

,cik_str,ticker,title
0,1045810,NVDA,NVIDIA CORP
1,320193,AAPL,Apple Inc.
2,1652044,GOOGL,Alphabet Inc.
3,789019,MSFT,MICROSOFT CORP
4,1018724,AMZN,AMAZON COM INC
...,...,...,...
10420,2086264,VHCPU,Vine Hill Capital Investment Corp. II
10421,2097364,TMTSU,Spartacus Acquisition Corp. II
10422,2107960,SWCPF,SWCC Corp/ADR
10423,2093654,ZJNGF,Zijin Gold International Co Limited/ADR


#### Test of $MSFT CIK lookup

In [6]:
msft_cik_ticker = cik_ticker[cik_ticker['ticker'] == 'MSFT']
print(msft_cik_ticker)
print("Extracted MSFT CIK: ", msft_cik_ticker['cik_str'].iloc[0])

  cik_str ticker           title
3  789019   MSFT  MICROSOFT CORP
Extracted MSFT CIK:  789019


### CIK by ticker _and_ exchange

#### Reformat the JSON to the form expected by pandas.read_json

In [7]:
def create_cik_exchange_json(filepath: str) -> str:
    """
    Formats the input company_tickers_exchange.json file to the form expected by pandas.read_json().
    """
    with open(filepath, 'r') as file:
        text = file.read().strip()
    cik_exchange = json.loads(text)
    assert validate_cik_exchange_data(cik_exchange['data'])
    return convert_cik_exchange_data_to_json(cik_exchange)

def validate_cik_exchange_data(data: dict, length: int=4) -> bool:
    """
    Validates that the input array of items in the CIK breakdown by ticker and exchange.
    """
    for item in data:
        if len(item) != length:
            print("Invalid length: ", item)
            return False
    return True

def convert_cik_exchange_data_to_json(unformatted_data: dict) -> str:
    """
    Converts the dict-ified version of the company_tickers_exchange.json data into a JSON string compatible with pandas.read_json().
    """
    fields = unformatted_data['fields']
    formatted = {}
    index = 0
    for item in unformatted_data['data']:
        formatted_item = {}
        for column in range(len(fields)):
            formatted_item[fields[column]] = item[column]
        formatted[index] = formatted_item
        index += 1
    return json.dumps(formatted)

#### Read in the CIK data labeled by ticker _and_ exchange

In [8]:
cik_ticker_exchange_json = create_cik_exchange_json('../../data/edgar/company_tickers_exchange.json')
cik_ticker_exchange = pd.read_json(StringIO(cik_ticker_exchange_json))  # raw string is deprecated
cik_ticker_exchange = cik_ticker_exchange.transpose()
cik_ticker_exchange


,cik,name,ticker,exchange
0,1045810,NVIDIA CORP,NVDA,Nasdaq
1,320193,Apple Inc.,AAPL,Nasdaq
2,1652044,Alphabet Inc.,GOOGL,Nasdaq
3,789019,MICROSOFT CORP,MSFT,Nasdaq
4,1018724,AMAZON COM INC,AMZN,Nasdaq
...,...,...,...,...
10407,2108124,"Odakyu Electric Railway Co., Ltd./ADR",ODERF,OTC
10408,2108180,NOF Corporation/ADR,NOFCF,OTC
10409,2108185,Azbil Corporation/ADR,YMATF,OTC
10410,2109234,BELIMO Holding AG/ADR,BLHWF,OTC


#### Test of $MSFT CIK lookup

In [9]:
msft_cik_ticker_exchange = cik_ticker_exchange[cik_ticker_exchange['ticker'] == 'MSFT']
print(msft_cik_ticker_exchange)
print("Extracted CIK: ", msft_cik_ticker_exchange['cik'].iloc[0])

      cik            name ticker exchange
3  789019  MICROSOFT CORP   MSFT   Nasdaq
Extracted CIK:  789019


## Company filing data

There are several endpoints, data dumps, directories, and search tools available.

The relevant API endpoints seem to be:
- `https://data.sec.gov/api/xbrl/companyconcept/CIK<cik_str>/<us-gaap|dei>/<company_concept_key>.json`
- `https://data.sec.gov/api/xbrl/companyfacts/CIK<cik_str>.json`

There are additional references to bulk zip files, presumably with similar underlying data (particularly for the xbrl/companyfacts.zip link):
- https://www.sec.gov/Archives/edgar/daily-index/xbrl/companyfacts.zip
- https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip

The EDGAR search can be accessed at https://www.sec.gov/edgar/search/# and allows for (manual) searching on `CIK`.  Additional fields like the start/end dates and text search can be leveraged to try and isolate particular filings.

Finally, the oldest filings seem to be most easily found directly from the `www.sec.gov/Archives/edgar/data/<CIK>` directory structure.  Notably, this directory _does not include leading 0s_ in the `CIK`.  For example, the path for $MSFT data is at https://www.sec.gov/Archives/edgar/data/789019/.  It's noted in [a page on accessing EDGAR data](https://www.sec.gov/search-filings/edgar-search-assistance/accessing-edgar-data) that each `CIK` directory (and its subdirectories) provide files to assist automated crawling of these directories for documents.  Traversal through `www.sec.gov/Archives/edgar/data/` without a provided `CIK`, however, is _not_ allowed.

### Padding CIKs

CIKs are meant to be 10-digit unique IDs.  Nevertheless, EDGAR is somewhat inconsistent on whether the leading 0s are needed for a particular path, and these leading 0s are not provided in EDGAR's provided ticker<->CIK mapping.  The function(s) below pad the front of an input CIK with 0s for convenience when required.

In [10]:
def pad_cik(cik: str | int) -> str:
    """
    Adds leading 0s to a CIK.
    """
    cik = str(cik)
    if len(cik) < 10:
        padding = 10 - len(cik)
        cik = ('0' * padding) + cik
    return cik

### $MSFT filings using `api/xrbl/companyfacts/` endpoint

In [22]:
msft_cik = msft_cik_ticker_exchange['cik'].iloc[0]
msft_cik = pad_cik(msft_cik)
msft_company_facts_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{msft_cik}.json"

company_facts_response = requests.get(msft_company_facts_url, headers=headers)
company_facts_response.status_code

200

#### Share history

I wanted to see how fine-grained the data here was, and how far back it went.  There are several keys related to shares outstanding and share repurchases, but they seem to only go back to 2007.  The filing dates appear to start in 2010, maybe suggesting some edits that were required to correct previous filings.

Search EDGAR manually, I seem to be able to find MSFT filings from calendar year 2000 (FY 2001), which include some comparisons to data from 1999.  E.g., https://www.sec.gov/Archives/edgar/data/789019/000103221001000273/0001032210-01-000273-0001.txt compares trailing 3- and 6-month fiscal data for the years ending Dec. 31st 1999 and 2000.

In [23]:
msft_facts = company_facts_response.json()

# Interesting keys:
# facts: {dei: {}, us-gaap: {}}
# dei: {EntityCommonStockSharesOutstanding}
# us-gaap: {StockRepurchasedDuringPeriodShares}
# us-gaap: {CommonStockSharesOutstanding}
# us-gaap: {WeightedAverageNumberOfSharesOutstandingBasic}
# us-gaap: {WeightedAverageNumberOfDilutedSharesOutstanding}
# us-gaap: {PaymentsForRepurchaseOfCommonStock}
# us-gaap: {StockRepurchasedDuringPeriodShares}
# us-gaap: {StockRepurchasedDuringPeriodValue}

first_entry = msft_facts['facts']['us-gaap']['CommonStockSharesOutstanding']['units']['shares'][0]
print("Values in the first entry:")
for key in first_entry:
    print(f"{key}: {first_entry[key]}")

Values in the first entry:
end: 2007-06-30
val: 9380000000
accn: 0001193125-10-171791
fy: 2010
fp: FY
form: 10-K
filed: 2010-07-30
frame: CY2007Q2I


### $MSFT filings using `api/xbrl/companyconcept/` endpoint

In [24]:
msft_cik = msft_cik_ticker_exchange['cik'].iloc[0]
msft_cik = pad_cik(msft_cik)
msft_company_concept_url = f"https://data.sec.gov/api/xbrl/companyconcept/CIK{msft_cik}/us-gaap/CommonStockSharesOutstanding.json"

company_concepts_response = requests.get(msft_company_concept_url, headers=headers)
company_concepts_response.status_code

200

#### Share history

It seems like the two API xrbl endpoints are using the same underlying data, which starts in 2007 despite EDGAR containing filings back to at least FY 2001.

In [26]:
msft_concepts = company_concepts_response.json()

first_entry = msft_concepts['units']['shares'][0]
print("Values in the first entry:")
for key in first_entry:
    print(f"{key}: {first_entry[key]}")

Values in the first entry:
end: 2007-06-30
val: 9380000000
accn: 0001193125-10-171791
fy: 2010
fp: FY
form: 10-K
filed: 2010-07-30
frame: CY2007Q2I
